In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
!nvidia-smi

Fri May  8 04:22:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 44.7 MB/s eta 0:00:00


In [12]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
token = secrets.get_secret("HF_W_TOKEN")
login(token=token)
print("Logged in")

Logged in


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "Qodo/PR-Review-Bench",
    data_files="git_code_review_bench_100_w_open_prs.jsonl"
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['repo', 'pr_url_to_review', 'issues', 'num_of_issues'],
        num_rows: 100
    })
})


In [6]:
def format_training_example(issue):
    code = issue.get('problematic_code_snippet', '').strip()
    title = issue.get('title', '').strip()
    description = issue.get('description', '').strip()
    rule = issue.get('rule_name', 'Unknown').strip()
    
    prompt = f"""### Code Review Task:
Analyze the following code snippet and identify the issue.

### Code:
{code}

### Review:
Title: {title}
Category: {rule}
Issue: {description}"""
    
    return prompt

formatted_data = []
for example in dataset['train']:
    for issue in example['issues']:
        snippet = issue.get('problematic_code_snippet', '')
        if snippet and snippet.strip():
            formatted_data.append({
                "text": format_training_example(issue),
                "repo": example['repo']
            })

print(f"Total training examples: {len(formatted_data)}")

Total training examples: 580


In [7]:
from datasets import Dataset

hf_dataset = Dataset.from_list(formatted_data)
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(split)

DatasetDict({
    train: Dataset({
        features: ['text', 'repo'],
        num_rows: 522
    })
    test: Dataset({
        features: ['text', 'repo'],
        num_rows: 58
    })
})


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "meta-llama/Llama-3.2-1B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda:0",
    token=token
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


In [9]:
import json
from datasets import Dataset

with open("/kaggle/working/combined_data.json", "r") as f:
    combined_data = json.load(f)

hf_dataset = Dataset.from_list(combined_data)
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])} Test: {len(split['test'])}")

Train: 1084 Test: 121


In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    args=SFTConfig(
        output_dir="/kaggle/working/pr-review-model-v2",
        num_train_epochs=5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
        dataset_text_field="text",
        max_length=512,
        packing=False,
    ),
)

print("Starting training on combined dataset...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1084 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1084 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/121 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/121 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Starting training on combined dataset...


Epoch,Training Loss,Validation Loss
1,1.604574,1.601786
2,1.544993,1.516999
3,1.425238,1.488371
4,1.339165,1.476320
5,1.258005,1.477017


TrainOutput(global_step=340, training_loss=1.467072520536535, metrics={'train_runtime': 5416.5088, 'train_samples_per_second': 1.001, 'train_steps_per_second': 0.063, 'total_flos': 8086171672363008.0, 'train_loss': 1.467072520536535})

In [13]:
trainer.model.save_pretrained("/kaggle/working/pr-review-adapter-v2")
tokenizer.save_pretrained("/kaggle/working/pr-review-adapter-v2")

trainer.model.push_to_hub("subbarayudu1234/pr-review-llama-1b-v2", token=token)
tokenizer.push_to_hub("subbarayudu1234/pr-review-llama-1b-v2", token=token)

print("Saved and pushed to HuggingFace")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved and pushed to HuggingFace


In [12]:
# Refresh token
secrets = UserSecretsClient()
token = secrets.get_secret("HF_W_TOKEN")
login(token=token)

# Push
trainer.model.push_to_hub("subbarayudu1234/pr-review-llama-1b", token=token)
tokenizer.push_to_hub("subbarayudu1234/pr-review-llama-1b", token=token)
print("Pushed to HuggingFace")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to HuggingFace


In [13]:
trainer.model.push_to_hub("subbarayudu1234/pr-review-llama-1b", token=token)
tokenizer.push_to_hub("subbarayudu1234/pr-review-llama-1b", token=token)
print("Pushed to HuggingFace")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to HuggingFace


In [14]:
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import json

openai_key = secrets.get_secret("OPENAI_API_KEY")
client = OpenAI(api_key=openai_key)

# Get all unique rule names from dataset
rule_names = list(set([
    issue.get('rule_name', 'Unknown')
    for example in dataset['train']
    for issue in example['issues']
    if issue.get('rule_name') and issue.get('rule_name') != 'Unknown'
]))

print(f"Total unique rules: {len(rule_names)}")
print("First 5 rules:")
for r in rule_names[:5]:
    print("-", r)

Total unique rules: 100
First 5 rules:
- React Components Must Sort JSX Props in Standard Order
- Async Methods Must Be Named with Async Suffix
- Backend Code Must Use Logging Instead of Print Statements
- All Source Files Must Include Tri-License Header
- Platform-Specific Code Must Use Appropriate Conditional Compilation


In [1]:
import os, json
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
from datasets import load_dataset

secrets = UserSecretsClient()
openai_key = secrets.get_secret("OPENAI_API_KEY")
client = OpenAI(api_key=openai_key)

# Reload dataset to get rule names
dataset = load_dataset("Qodo/PR-Review-Bench", data_files="git_code_review_bench_100_w_open_prs.jsonl")

rule_names = list(set([
    issue.get('rule_name', 'Unknown')
    for example in dataset['train']
    for issue in example['issues']
    if issue.get('rule_name') and issue.get('rule_name') != 'Unknown'
]))

additional_rules = [
    "Functions Must Have Single Responsibility",
    "Magic Numbers Must Be Named Constants",
    "Error Messages Must Be User Friendly",
    "API Endpoints Must Be Versioned",
    "Passwords Must Never Be Logged",
    "Database Connections Must Use Connection Pooling",
    "All User Input Must Be Sanitized",
    "Functions Must Not Exceed 20 Lines",
    "Variables Must Have Descriptive Names",
    "Dead Code Must Be Removed",
    "Null Checks Must Be Explicit",
    "Async Functions Must Handle Errors",
    "API Responses Must Include Status Codes",
    "Sensitive Data Must Be Encrypted at Rest",
    "Tests Must Cover Edge Cases",
    "Dependencies Must Be Pinned to Specific Versions",
    "Global Variables Must Be Avoided",
    "Comments Must Explain Why Not What",
    "File Names Must Be Consistent with Class Names",
    "All Exceptions Must Be Caught and Handled",
    "SQL Queries Must Use Parameterized Statements",
    "Memory Must Be Freed After Allocation",
    "Recursive Functions Must Have Base Cases",
    "Configuration Must Not Be Hardcoded",
    "Logging Must Include Context Information",
]

all_rules = rule_names + additional_rules
print(f"Total rules: {len(all_rules)}")

README.md: 0.00B [00:00, ?B/s]

(…)t_code_review_bench_100_w_open_prs.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total rules: 125


In [2]:
def generate_synthetic_example(rule_name):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": "You are a senior code reviewer. Generate realistic code review examples."
            },
            {
                "role": "user",
                "content": f"""Generate a realistic code review example for this rule: "{rule_name}"

Return ONLY a JSON object with exactly these fields:
{{
    "code_snippet": "the problematic code (5-15 lines, realistic)",
    "title": "short title of the issue",
    "description": "detailed explanation of why this violates the rule and how to fix it"
}}"""
            }
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

synthetic_data = []
failed = 0

for i, rule in enumerate(all_rules):
    for attempt in range(5):
        try:
            example = generate_synthetic_example(rule)
            synthetic_data.append({
                "text": f"""### Code Review Task:
Analyze the following code snippet and identify the issue.

### Code:
{example['code_snippet']}

### Review:
Title: {example['title']}
Category: {rule}
Issue: {example['description']}""",
                "repo": "synthetic"
            })
        except Exception as e:
            failed += 1
            continue

    # Save every 10 rules in case of interruption
    if (i + 1) % 10 == 0:
        with open("/kaggle/working/synthetic_data.json", "w") as f:
            json.dump(synthetic_data, f)
        print(f"Progress: {i+1}/{len(all_rules)} rules — {len(synthetic_data)} examples saved")

# Final save
with open("/kaggle/working/synthetic_data.json", "w") as f:
    json.dump(synthetic_data, f)

print(f"\nTotal synthetic examples: {len(synthetic_data)}")
print(f"Failed: {failed}")
print("Saved to /kaggle/working/synthetic_data.json")

Progress: 10/125 rules — 50 examples saved
Progress: 20/125 rules — 100 examples saved
Progress: 30/125 rules — 150 examples saved
Progress: 40/125 rules — 200 examples saved
Progress: 50/125 rules — 250 examples saved
Progress: 60/125 rules — 300 examples saved
Progress: 70/125 rules — 350 examples saved
Progress: 80/125 rules — 400 examples saved
Progress: 90/125 rules — 450 examples saved
Progress: 100/125 rules — 500 examples saved
Progress: 110/125 rules — 550 examples saved
Progress: 120/125 rules — 600 examples saved

Total synthetic examples: 625
Failed: 0
Saved to /kaggle/working/synthetic_data.json


In [3]:
# Reload real data
def format_training_example(issue):
    code = issue.get('problematic_code_snippet', '').strip()
    title = issue.get('title', '').strip()
    description = issue.get('description', '').strip()
    rule = issue.get('rule_name', 'Unknown').strip()
    return f"""### Code Review Task:
Analyze the following code snippet and identify the issue.

### Code:
{code}

### Review:
Title: {title}
Category: {rule}
Issue: {description}"""

formatted_data = []
for example in dataset['train']:
    for issue in example['issues']:
        snippet = issue.get('problematic_code_snippet', '')
        if snippet and snippet.strip():
            formatted_data.append({
                "text": format_training_example(issue),
                "repo": example['repo']
            })

# Combine
combined_data = formatted_data + synthetic_data

# Save
with open("/kaggle/working/combined_data.json", "w") as f:
    json.dump(combined_data, f)

print(f"Real examples: {len(formatted_data)}")
print(f"Synthetic examples: {len(synthetic_data)}")
print(f"Total combined: {len(combined_data)}")
print("Saved to /kaggle/working/combined_data.json")

Real examples: 580
Synthetic examples: 625
Total combined: 1205
Saved to /kaggle/working/combined_data.json
